# 2.3 重试、退避与熔断：让失败「可见、可控、可恢复」

**学完你能**：给工具调用装上指数退避重试与熔断器；说出哪些错误值得重试；用测试证明熔断真的快速失败。

Sources: agent-expert-outline.md 思维 #6/#9 · agent/tools/runner.py（实现本体）· 错误分类对照 OpenAI API 错误码（429/5xx 可重试，400 不可）[未实访] · 课前自修：Missing Semester「Debugging and Profiling」

## 概念导读：重试的三条铁律

1. **分类重试**：只有瞬态错误（网络抖动/限流/超时）值得重试；参数错（400 类）重试一万次也不会好。
2. **退避有上限**：指数退避（0.2→0.4→0.8 秒）+ max_attempts，两者缺一就是死循环反例。
3. **熔断失败关闭**：连续失败达到阈值 → 直接拒绝调用（快速失败），冷却后半开试探——宁可这一步暂停，不可拖死整个循环。

> 反模式预告：`while True: retry()` 没有退避没有上限——练习 2 会让你亲手造一个再拆掉。

In [ ]:
# 注入 flaky 工具（前 N 次失败），验证重试与分类——零成本
from agent.tools.runner import FatalToolError, RetryPolicy, ToolRunner, TransientToolError

def flaky(fail_times: int, exc=TransientToolError):
    state = {"calls": 0}
    def fn(_: str) -> str:
        state["calls"] += 1
        if state["calls"] <= fail_times:
            raise exc("模拟故障")
        return "ok"
    fn.calls = state
    return fn

policy = RetryPolicy(max_attempts=3, base_delay=0.01, cooldown=0.05, threshold=3)
r = ToolRunner(policy=policy)

f = flaky(2)
print("瞬态×2 后成功:", r.run("t", f, "x"), "| 调用次数:", f.calls["calls"])

g = flaky(99, exc=FatalToolError)
print("致命错误:", r.run("t", g, "x"), "| 调用次数:", g.calls["calls"], "（一次都不值得重试）")

In [ ]:
# 熔断：连续失败达阈值 → 第 4 次调用直接 [熔断]，工具根本没被执行（快速失败）
r2 = ToolRunner(policy=policy)
bad = flaky(99)
for i in range(3):
    r2.run("t", bad, "x")
before = bad.calls["calls"]
print("第 4 次调用:", r2.run("t", bad, "x"))
print("工具被真实执行了吗:", bad.calls["calls"] > before, "（False = 熔断拦截成功）")

import time
time.sleep(policy.cooldown + 0.02)   # 冷却结束 → 半开试探
good = flaky(0)
print("半开试探:", r2.run("t", good, "x"), "→ 熔断关闭，恢复正常")

In [ ]:
# 装配：wrap 后直接进 ReAct 循环——保护逻辑对循环透明
from agent.react import run_react

r3 = ToolRunner(policy=policy)
script = iter(["Thought: 调用\nAction: flaky hi", "Thought: 完成\nFinish: done"])
result = run_react("x", {"flaky": r3.wrap("flaky", flaky(1))}, llm_fn=lambda m: next(script), max_steps=3)
print("循环正常收尾:", result["answer"] == "done", "| 步骤:", result["steps"])

## 练习（答案只进 solutions 子仓）

1. **改参数**：把 max_attempts 改成 1、threshold 改成 1，重跑熔断演示——观察「一次失败即熔断」在什么场景下是对的（提示：上游已限流时）。
2. **破坏性实验**：亲手写一个 `while True: retry()` 的死循环反例（用 try/KeyboardInterrupt 保护着跑 3 秒），数一数它烧了多少次调用——然后对照 runner 的三个参数说出它缺了什么。
3. **扩展题（挑战档）**：给 ToolRunner 加「半开状态连续失败立即重开」语义（当前实现：半开一次成功即关闭），先写失败测试。

## 提交前检查

```bash
uv run pytest -q && uvx ruff check . && uv run python smoke.py
jupyter nbconvert --clear-output --inplace ch02-tools/2.3-retry-and-errors.ipynb
```